## 1. Bibliotecas e Dados

In [47]:
!uv pip install "numpy==1.26.4" "spacy==3.7.4" pyvis pandas nltk
!python -m spacy download en_core_web_sm
!uv pip install pyvis

Using Python 3.11.13 environment at: /usr
Audited 5 packages in 145ms
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 40.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Using Python 3.11.13 environment at: /usr
Audited 1 package in 130ms


In [48]:
import spacy
import pandas as pd
import re
import nltk
from pathlib import Path
from spacy.matcher import Matcher
from spacy.util import filter_spans
from nltk.corpus import stopwords

from pyvis.network import Network
import IPython


ROOT = Path('')
data = pd.read_csv(ROOT / 'cases.csv')
metadata = pd.read_csv(ROOT / 'metadata.csv')

In [49]:
# merge e seleção do text
full_data = pd.merge(data, metadata)
full_data = full_data[['case_text', 'gender', 'case_id', 'major_mesh_terms', 'mesh_terms']]
text = full_data['case_text'][0]

# modelo spaCy
nlp = spacy.load('en_core_web_sm')
doc = nlp(text)

nltk.download('stopwords', quiet=True)

True

## 2. Captura de Medidas (Regex)

In [50]:
measurement_pattern = re.compile(r'(\d+(?:,\d+)?(?:\.\d+)?)\s*(cm|mm|ng/ml|iu/ml|mg)')
measurements = []

for match in measurement_pattern.finditer(text):
    value, unit = match.group(1), match.group(2)
    measurements.append({
        'node_type': 'ExamResult',
        'label_original': match.group(0),
        'label_normalizado': f"{value} {unit}",
        'token_start': -1,
        'token_end': -1,
        'span_start': match.start(),
        'span_end': match.end(),
        'value': value,
        'unit': unit
    })

measurements_df = pd.DataFrame(measurements)
display(measurements_df.head())

,node_type,label_original,label_normalizado,token_start,token_end,span_start,span_end,value,unit
0,ExamResult,6cm,6 cm,-1,-1,337,340,6,cm
1,ExamResult,6cm,6 cm,-1,-1,516,519,6,cm
2,ExamResult,9cm,9 cm,-1,-1,522,525,9,cm
3,ExamResult,"12,476.5ng/ml","12,476.5 ng/ml",-1,-1,777,790,"12,476.5",ng/ml
4,ExamResult,6iu/ml,6 iu/ml,-1,-1,837,843,6,iu/ml


## 3. Extração de Entidades (scispaCy + Matcher) e Unificação de Nós

### Identificar paciente

In [51]:
extracted_entities = []

patient_terms = ["woman", "man", "girl", "boy", "male", "female", "patient"]
patient_matcher = Matcher(nlp.vocab)
patient_matcher.add("PATIENT_DEMO", [[{"LOWER": {"IN": patient_terms}}]])
patient_matches = patient_matcher(doc)

patient_label = "patient"
if patient_matches:
    _, start, end = patient_matches[0]
    span = doc[start:end]
    patient_label = span.text.lower()
    extracted_entities.append({
        'node_type': 'Patient',
        'label_original': span.text,
        'label_normalizado': patient_label,
        'token_start': start, 'token_end': end,
        'span_start': span.start_char, 'span_end': span.end_char,
        'value': None, 'unit': None
    })
PATIENT_NODE_LABEL = patient_label

### Matcher para resgatar exames, procedimentos e conceitos clínicos

In [52]:
disease_keywords = ['pain', 'nausea', 'constipation', 'malignancy', 'neoplasm', 'bleeding', 'cyst', 'lesion', 'abnormality', 'gdc']
exam_keywords = ['tomography', 'endoscopy', 'resection', 'pancreatectomy', 'fna', 'examination', 'exploration', 'pathology']

stop_words_nltk = set(stopwords.words('english'))
custom_stops = {'day', 'history', 'month', 'year', 'time', 'presence', 'evidence', 'fig', 'figure'}
all_stopwords = stop_words_nltk.union(custom_stops)

concept_matcher = Matcher(nlp.vocab)
# Padrões: Adjetivo(s) opcional(is) seguido(s) de Substantivo(s)
concept_matcher.add("CLINICAL_CONCEPT", [
    [{"POS": "ADJ", "OP": "*"}, {"POS": "NOUN", "OP": "+"}],
    [{"POS": "PROPN", "OP": "+"}] # Pega siglas como GDC ou CEA
])

# Filtra sobreposições (ex: pega "abdominal pain" inteiro em vez de separar "abdominal" e "pain")
spans = [doc[start:end] for _, start, end in concept_matcher(doc)]
filtered_spans = filter_spans(spans)

for span in filtered_spans:

    valid_tokens = [
        t for t in span
        if t.lemma_.lower() not in all_stopwords
        and not t.is_punct
        and not t.is_digit
    ]

    if not valid_tokens:
        continue

    clean_label = " ".join([t.lemma_.lower() for t in valid_tokens])

    # Ignora o paciente e palavras comuns irrelevantes
    if PATIENT_NODE_LABEL in clean_label:
        continue

    # Classificação baseada no dicionário (Substitui o scispaCy)
    node_class = 'MedicalConcept' # Tipo genérico padrão
    if any(word in clean_label for word in disease_keywords):
        node_class = 'DISEASE'
    elif any(word in clean_label for word in exam_keywords):
        node_class = 'Procedure/Exam'

    extracted_entities.append({
        'node_type': node_class,
        'label_original': span.text,
        'label_normalizado': span.lemma_.lower(),
        'token_start': span.start, 'token_end': span.end,
        'span_start': span.start_char, 'span_end': span.end_char,
        'value': None, 'unit': None
    })

entities_df = pd.DataFrame(extracted_entities).drop_duplicates(subset=['label_normalizado']).reset_index(drop=True)
display(entities_df.head(15))

,node_type,label_original,label_normalizado,token_start,token_end,span_start,span_end,value,unit
0,Patient,woman,woman,6,7,14,19,None,None
1,MedicalConcept,right flank,right flank,15,17,54,65,None,None
2,MedicalConcept,lower quadrant,low quadrant,18,20,70,84,None,None
3,DISEASE,abdominal pain,abdominal pain,20,22,85,99,None,None
4,DISEASE,nausea,nausea,24,25,116,122,None,None
5,DISEASE,constipation,constipation,26,27,127,139,None,None
6,MedicalConcept,family,family,32,33,159,165,None,None
7,MedicalConcept,medication history,medication history,34,36,170,188,None,None
8,Procedure/Exam,physical examination,physical examination,43,45,229,249,None,None
9,MedicalConcept,contrast,contrast,50,51,282,290,None,None


### Tabela de Nós Final

In [54]:
entities_df = pd.DataFrame(extracted_entities).drop_duplicates(subset=['label_normalizado']).reset_index(drop=True)
final_nodes_df = pd.concat([entities_df, measurements_df], ignore_index=True)
print(f"TABELA DE NÓS")
display(final_nodes_df.head(15))

TABELA DE NÓS


,node_type,label_original,label_normalizado,token_start,token_end,span_start,span_end,value,unit
0,Patient,woman,woman,6,7,14,19,None,None
1,MedicalConcept,right flank,right flank,15,17,54,65,None,None
2,MedicalConcept,lower quadrant,low quadrant,18,20,70,84,None,None
3,DISEASE,abdominal pain,abdominal pain,20,22,85,99,None,None
4,DISEASE,nausea,nausea,24,25,116,122,None,None
5,DISEASE,constipation,constipation,26,27,127,139,None,None
6,MedicalConcept,family,family,32,33,159,165,None,None
7,MedicalConcept,medication history,medication history,34,36,170,188,None,None
8,Procedure/Exam,physical examination,physical examination,43,45,229,249,None,None
9,MedicalConcept,contrast,contrast,50,51,282,290,None,None


## 4. Geração do Grafo de Conhecimento (Tabela de Arestas)

In [55]:
edges = []
edge_id_counter = 1

verb_relations = {
    ("present", "have", "experience", "associate"): "HAS_SYMPTOM",
    ("undergo", "perform", "do", "plan", "convert"): "UNDERWENT_PROCEDURE",
    ("reveal", "demonstrate", "suggest", "show", "note", "observe"): "SUPPORTS",
    ("treat", "resect", "discharge"): "TREATED_BY"
}

### Arestas Sintáticas

In [56]:
for sent in doc.sents:
    # pega apenas os nós que estão nesta frase
    nodes_in_sent = [row for _, row in entities_df.iterrows() if row['node_type'] != 'Patient' and sent.start <= row['token_start'] < sent.end]

    for token in sent:
        if token.pos_ == "VERB":
            verbo_lema = token.lemma_.lower()

            for verbs, rel in verb_relations.items():
                if verbo_lema in verbs:
                    if rel in ["HAS_SYMPTOM", "UNDERWENT_PROCEDURE", "TREATED_BY"]:
                        for node in nodes_in_sent:
                            # filtra um pouco
                            if rel == "HAS_SYMPTOM" and node['node_type'] not in ['DISEASE', 'MedicalConcept']: continue
                            if rel == "UNDERWENT_PROCEDURE" and node['node_type'] not in ['Procedure/Exam']: continue

                            edges.append({
                                'source_label': PATIENT_NODE_LABEL,
                                'target_label': node['label_normalizado'],
                                'relation': rel
                            })

                    elif rel == "SUPPORTS" and len(nodes_in_sent) >= 2:
                        edges.append({
                            'source_label': nodes_in_sent[0]['label_normalizado'],
                            'target_label': nodes_in_sent[1]['label_normalizado'],
                            'relation': rel
                        })
                    break

edges_df = pd.DataFrame(edges)
display(edges_df.head())

,source_label,target_label,relation
0,woman,right flank,HAS_SYMPTOM
1,woman,low quadrant,HAS_SYMPTOM
2,woman,abdominal pain,HAS_SYMPTOM
3,woman,nausea,HAS_SYMPTOM
4,woman,constipation,HAS_SYMPTOM


### Arestas de Valores por distância de caracteres

In [57]:
for _, ent_row in entities_df.iterrows():
    if ent_row['node_type'] == 'Patient': continue
    for _, meas_row in measurements_df.iterrows():
        if abs(ent_row['span_start'] - meas_row['span_start']) < 30:
            edges.append({
                'source_label': ent_row['label_normalizado'],
                'target_label': meas_row['label_normalizado'],
                'relation': 'HAS_VALUE'
            })

final_graph_edges_df = pd.DataFrame(edges).drop_duplicates(subset=['source_label', 'target_label', 'relation']).reset_index(drop=True)

## Grafo de Conhecimento final

In [58]:
final_graph_edges_df.insert(0, 'edge_id', [f"E_{i+1:03d}" for i in range(len(final_graph_edges_df))])

display(final_graph_edges_df.head(20))

,edge_id,source_label,target_label,relation
0,E_001,woman,right flank,HAS_SYMPTOM
1,E_002,woman,low quadrant,HAS_SYMPTOM
2,E_003,woman,abdominal pain,HAS_SYMPTOM
3,E_004,woman,nausea,HAS_SYMPTOM
4,E_005,woman,constipation,HAS_SYMPTOM
5,E_006,woman,computed tomography,UNDERWENT_PROCEDURE
6,E_007,contrast,computed tomography,SUPPORTS
7,E_008,woman,fna,UNDERWENT_PROCEDURE
8,E_009,eus,fna,SUPPORTS
9,E_010,woman,eus,HAS_SYMPTOM


# Vizualização do Grafo (Com LLM)

In [59]:

# Inicializa a rede interativa configurada para o Colab
net = Network(notebook=True, directed=True, cdn_resources='remote', height="800px", width="100%")

# Dicionário de cores para diferenciar visualmente o tipo de informação no grafo
color_map = {
    'Patient': '#ff4d4d',         # Vermelho
    'DISEASE': '#ffa64d',         # Laranja
    'CHEMICAL': '#79d2a6',        # Verde
    'Procedure/Exam': '#66b3ff',  # Azul
    'MedicalConcept': '#d9d9d9',  # Cinza
    'ExamResult': '#ffff66'       # Amarelo
}

# 1. Adicionando os Nós (Nodes)
for _, row in final_nodes_df.iterrows():
    node_id = row['label_normalizado']
    node_type = row['node_type']

    # Define a cor baseada no tipo, ou cinza claro por padrão
    node_color = color_map.get(node_type, '#e6e6e6')

    # Texto que aparece quando passa o mouse por cima
    hover_text = f"Tipo: {node_type}"
    if pd.notna(row.get('value')):
        hover_text += f" | Valor: {row['value']} {row['unit']}"

    net.add_node(node_id, label=node_id, title=hover_text, color=node_color)

# 2. Adicionando as Arestas (Edges/Relações)
# Criamos um conjunto de nós existentes para evitar erros caso a aresta aponte para um nó filtrado
existing_nodes = set(net.get_nodes())

for _, row in final_graph_edges_df.iterrows():
    source = row['source_label']
    target = row['target_label']
    relation = row['relation']

    if source in existing_nodes and target in existing_nodes:
        # Adiciona a seta com o rótulo da relação
        net.add_edge(source, target, title=relation, label=relation, color="#808080")

# 3. Física e Layout (Para o grafo não ficar todo embolado)
net.set_options("""
var options = {
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -50,
      "centralGravity": 0.01,
      "springLength": 100,
      "springConstant": 0.08
    },
    "minVelocity": 0.75,
    "solver": "forceAtlas2Based"
  },
  "edges": {
    "font": {
      "size": 10,
      "align": "middle"
    },
    "arrows": {
      "to": {"enabled": true, "scaleFactor": 0.5}
    }
  }
}
""")

# Gera o arquivo HTML e renderiza na célula do Colab
html_file = "knowledge_graph.html"
net.show(html_file)
IPython.display.HTML(filename=html_file)

knowledge_graph.html
